In [0]:
%run /Workspace/Users/riley.day@kwa-analytics.com/Databricks-Certified-Data-Engineer-Associate/Includes/Copy-Datasets

In [0]:
files = dbutils.fs.ls(f"{dataset_bookstore}/orders-raw")
display(files)

In [0]:
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "dbfs:/mnt/demo/orders_checkpoint")
    .load(f"{dataset_bookstore}/orders-raw")
    .createOrReplaceTempView("orders_raw_temp")
)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW orders_tmp AS (
    SELECT *, current_timestamp() as arrival_time,
    input_file_name() as source_file
    from orders_raw_temp
)

In [0]:
%sql
select * from orders_tmp

In [0]:
(spark.table("orders_tmp")
 .writeStream
 .format("delta")
 .outputMode("append")
 .option("checkpointLocation", "dbfs:/mnt/demo/orders_bronze")
 .table("orders_bronze")
 )

In [0]:
%sql
select count(*) from orders_bronze

In [0]:
load_new_data()

In [0]:
(spark.read
    .format("json")
    .load(f"{dataset_bookstore}/customers-json")
    .createOrReplaceTempView("customers_lookup"))

In [0]:
%sql
select * from customers_lookup

In [0]:
(spark.readStream
    .table("orders_bronze")
    .createOrReplaceTempView("orders_bronze_tmp"))

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW orders_enriched_tmp AS (
    SELECT order_id, quantity, o.customer_id, c.profile:first_name as f_name, c.profile:last_name as l_name, cast(from_unixtime(order_timestamp, 'yyyy-MM-dd HH:mm:ss') as timestamp) as order_timestamp, books
    FROM orders_bronze_tmp o
    INNER JOIN customers_lookup c
    ON o.customer_id = c.customer_id
    WHERE quantity >0
)

In [0]:
(spark.table("orders_enriched_tmp")
 .writeStream
 .format("delta")
 .outputMode("append")
 .option("checkpointLocation", "dbfs:/mnt/demo/checkpoints/orders_silver")
 .table("orders_silver"))

In [0]:
%sql
select * from orders_silver

In [0]:
%sql
select count(*) from orders_silver

In [0]:
load_new_data()

In [0]:
(spark.readStream
    .table("orders_silver")
    .createOrReplaceTempView("orders_silver_tmp"))

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW daily_customer_books_tmp AS (
    select customer_id, f_name, l_name, date_trunc("DD", order_timestamp) as order_date, sum(quantity) as books_count from orders_silver_tmp
    group by customer_id, f_name, l_name, date_trunc("DD", order_timestamp)
)

In [0]:
%sql
select * from daily_customer_books_tmp

In [0]:
(spark.table("daily_customer_books_tmp")
    .writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", "dbfs:/mnt/demo/checkpoints/daily_customers_books")
    .trigger(availableNow=True)
    .table("daily_customer_books")
    )

In [0]:
%sql
select * from daily_customer_books

In [0]:
load_new_data(all=True)

In [0]:
for s in spark.streams.active:
    print("Stropping Stream: " + s.id)
    s.stop()
    s.awaitTermination()